# TD 2 — Step 2: The wrong consensus (Pair B)

Workflow:
0. **Configuration**: fill in paths, column names, the question and the panel
1. Load the traces and find the shared wrong values
2. Reference value and gap in cents
3. 2-of-3 and 3-of-5 votes
4. SQL queries of the agreeing models, run on `data/boutique.sqlite`
5. Diagnosis of the faulty clause (written by you)
6. Delta query + check that it accounts for the gap to the cent
7. Independent check of the reference
8. DEMO-VOTE (toy example, required)
9. Fallback case DEMO-SQL-C11 (if you have no case of your own)
10. Summary to copy into `reports/td2.md`

Cells marked **✏️ TO FILL** need your input.

## 0. Configuration ✏️ TO FILL

In [98]:
import sqlite3
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 200)

# ---- Files -------------------------------------------------------------------
TRACES_CSV = "../../td1/traces-td1.csv"
REFERENCE_CSV = "../../td1/reference-td1.csv"
DB_PATH = "../../td1/data/boutique.sqlite"

# ---- Columns of traces-td1.csv (check them in cell 1) ------------------------
COL_QUESTION = "case_id"
COL_MODEL = "model_requested"
COL_VALUE = "value"
COL_VERDICT = "verdict"
COL_SQL = "sql"            # column holding the generated SQL; None if absent
WRONG_LABEL = "incorrect"

# ---- Columns of reference-td1.csv --------------------------------------------
REF_COL_QUESTION = "case_id"
REF_COL_VALUE = "revenue_cents"

# ---- Your case ---------------------------------------------------------------
QUESTION = "C11"

# Fixed 2-of-3 panel imposed by atelier-vote.py (three distinct families; Dots is NOT a member)
PANEL_MODELS = ["openai/gpt-oss-120b", "cohere/north-mini-code:free", "qwen/qwen3.8-27b"]
# The 5 TD 1 models: leave None to take every model present in the traces
ALL_MODELS = None

## 1. Load the traces and find the shared wrong values

In [99]:
def normalize(value) -> str:
    """'0', '0.0' and ' 0 ' become '0'; non-numbers are only stripped."""
    if pd.isna(value):
        return ""
    v = str(value).strip()
    try:
        f = float(v)
        return str(int(f)) if f.is_integer() else str(f)
    except ValueError:
        return v


traces = pd.read_csv(TRACES_CSV, dtype=str)
print("Columns:", list(traces.columns))
print(len(traces), "rows")
print("Verdicts:", sorted(traces[COL_VERDICT].dropna().unique()))
print("Models:", sorted(traces[COL_MODEL].dropna().unique()))

missing = [c for c in (COL_QUESTION, COL_MODEL, COL_VALUE, COL_VERDICT) if c not in traces.columns]
assert not missing, f"Missing columns {missing}: fix cell 0"

traces["value_norm"] = traces[COL_VALUE].map(normalize)
traces["is_wrong"] = traces[COL_VERDICT].str.strip().str.lower() == WRONG_LABEL

if ALL_MODELS is None:
    ALL_MODELS = sorted(traces[COL_MODEL].dropna().unique())
print("\nModels used for 3-of-5:", ALL_MODELS)
if len(ALL_MODELS) != 5:
    print(f"⚠️ {len(ALL_MODELS)} models found instead of 5: check the traces / ALL_MODELS")

Columns: ['cell_id', 'provider', 'model_requested', 'model_returned', 'case_id', 'question', 'prompt_sha256', 'sql', 'value', 'reference', 'verdict', 'verdict_detail', 'http_status', 'error', 'latency_ms', 'tokens_prompt', 'tokens_completion', 'tokens_reasoning', 'cost_native', 'attempts', 'date_utc', 'nature', 'source_file']
126 rows
Verdicts: ['correct', 'incorrect', 'not_executed', 'sql_invalid', 'sql_rejected']
Models: ['cohere/north-mini-code:free', 'dots-studio/dots-3-note-preview:free', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3.8-27b']

Models used for 3-of-5: ['cohere/north-mini-code:free', 'dots-studio/dots-3-note-preview:free', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3.8-27b']


In [100]:
wrong = traces[traces["is_wrong"] & (traces["value_norm"] != "")]
shared = (
    wrong.groupby([COL_QUESTION, "value_norm"])[COL_MODEL]
    .agg(n_models="nunique", models=lambda s: sorted(s.unique()))
    .reset_index()
)
shared = shared[shared["n_models"] >= 2].sort_values("n_models", ascending=False)

if shared.empty:
    print("No shared wrong value -> use the DEMO-SQL-C11 case (section 9).")
else:
    print("Shared wrong values (>= 2 models):")
    display(shared)

Shared wrong values (>= 2 models):


,case_id,value_norm,n_models,models
0,C02,0,3,"[cohere/north-mini-code:free, dots-studio/dots-3-note-preview:free, openai/gpt-oss-20b]"
3,C09,0,2,"[cohere/north-mini-code:free, qwen/qwen3.8-27b]"
6,C11,0,2,"[cohere/north-mini-code:free, dots-studio/dots-3-note-preview:free]"


## 2. The chosen question: answers, reference, gap

In [101]:
q_rows = traces[traces[COL_QUESTION] == QUESTION]
cols = [COL_MODEL, COL_VALUE, "value_norm", COL_VERDICT]
display(q_rows[cols])

dupes = q_rows[COL_MODEL].value_counts()
if (dupes > 1).any():
    print("⚠️ Several rows for the same model on this question:", dupes[dupes > 1].to_dict())
    print("   Decide which run counts before voting (the first one is used below).")

# Dots has two runs on C11: traces-E.csv (13:49:44 UTC, value 0) and traces-E-Mathias.csv
# (13:48:22 UTC, value 446690). Rule of atelier-vote.py: the LAST attempt counts -> traces-E.csv,
# which is also the first row in traces-td1.csv, so the value used below (0) is the right one.
DOTS_RUNS = q_rows[q_rows[COL_MODEL].str.startswith("dots")][[COL_VALUE, "source_file", "date_utc"]]
display(DOTS_RUNS.sort_values("date_utc"))

ref = pd.read_csv(REFERENCE_CSV, dtype=str)
ref_row = ref[ref[REF_COL_QUESTION] == QUESTION]
assert len(ref_row) == 1, f"Reference not found (or duplicated) for {QUESTION}"
REFERENCE = normalize(ref_row[REF_COL_VALUE].iloc[0])

q_wrong = shared[shared[COL_QUESTION] == QUESTION] if not shared.empty else shared
if len(q_wrong):
    WRONG_VALUE = q_wrong["value_norm"].iloc[0]
    AGREEING_MODELS = q_wrong["models"].iloc[0]
else:
    # ✏️ fill in by hand if the case doesn't come from your traces (e.g. DEMO-SQL-C11)
    WRONG_VALUE = "0"
    AGREEING_MODELS = ["openai/gpt-oss-20b", "cohere/north-mini-code:free"]
    print("ℹ️ No shared wrong value in your traces for this question: hand-filled values used.")

GAP_CENTS = int(float(REFERENCE)) - int(float(WRONG_VALUE))
print(f"Reference = {REFERENCE} | shared wrong value = {WRONG_VALUE} | models = {AGREEING_MODELS}")
print(f"Gap (reference - wrong) = {GAP_CENTS} cents")

,model_requested,value,value_norm,verdict
11,openai/gpt-oss-120b,446690,446690,correct
31,openai/gpt-oss-20b,446690,446690,correct
51,cohere/north-mini-code:free,0,0,incorrect
71,qwen/qwen3.8-27b,446690,446690,correct
97,dots-studio/dots-3-note-preview:free,0,0,incorrect
116,dots-studio/dots-3-note-preview:free,446690,446690,correct


⚠️ Several rows for the same model on this question: {'dots-studio/dots-3-note-preview:free': 2}
   Decide which run counts before voting (the first one is used below).


,value,source_file,date_utc
116,446690,traces-E-Mathias.csv,2026-10-21 13:48:22
97,0,traces-E.csv,2026-10-21 13:49:44


Reference = 446690 | shared wrong value = 0 | models = ['cohere/north-mini-code:free', 'dots-studio/dots-3-note-preview:free']
Gap (reference - wrong) = 446690 cents


## 3. Votes: 2-of-3 panel and 3-of-5 quorum

In [102]:
def vote(models, k, label):
    votes = {}
    for m in models:
        r = q_rows[q_rows[COL_MODEL] == m]
        votes[m] = r["value_norm"].iloc[0] if len(r) else None  # None = no answer
    counts = Counter(v for v in votes.values() if v not in (None, ""))
    winners = [v for v, c in counts.items() if c >= k]
    winner = winners[0] if winners else None

    if winner is None:
        outcome = "NO QUORUM: no value reaches the threshold"
    elif winner == WRONG_VALUE:
        outcome = "the WRONG value wins (wrong consensus accepted)"
    elif winner == REFERENCE:
        outcome = "the REFERENCE wins (wrong value outvoted)"
    else:
        outcome = "ANOTHER value wins (also different from the reference)"

    print(f"--- {label}: threshold k = {k} of n = {len(models)} ---")
    for m, v in votes.items():
        print(f"  {m:45s} -> {v}")
    print("  Count:", dict(counts))
    print("  Result:", winner, "=>", outcome, "\n")
    return votes, winner, outcome


missing_panel = [m for m in PANEL_MODELS if m not in set(traces[COL_MODEL])]
if missing_panel:
    print("⚠️ Panel models not found in the traces (spelling?):", missing_panel, "\n")

VOTE_2_3 = vote(PANEL_MODELS, 2, "Fixed panel 2-of-3")
VOTE_3_5 = vote(ALL_MODELS, 3, "Quorum 3-of-5")

--- Fixed panel 2-of-3: threshold k = 2 of n = 3 ---
  openai/gpt-oss-120b                           -> 446690
  cohere/north-mini-code:free                   -> 0
  qwen/qwen3.8-27b                              -> 446690
  Count: {'446690': 2, '0': 1}
  Result: 446690 => the REFERENCE wins (wrong value outvoted) 

--- Quorum 3-of-5: threshold k = 3 of n = 5 ---
  cohere/north-mini-code:free                   -> 0
  dots-studio/dots-3-note-preview:free          -> 0
  openai/gpt-oss-120b                           -> 446690
  openai/gpt-oss-20b                            -> 446690
  qwen/qwen3.8-27b                              -> 446690
  Count: {'0': 2, '446690': 3}
  Result: 446690 => the REFERENCE wins (wrong value outvoted) 



## 4. SQL queries of the agreeing models ✏️ CHECK

The cell below tries to pull the SQL from the traces. If `COL_SQL` is absent, paste the queries
by hand into `FAULTY_QUERIES`. Then write **your reference query** in `REFERENCE_QUERY`.

In [103]:
FAULTY_QUERIES = {}
if COL_SQL and COL_SQL in traces.columns:
    for m in AGREEING_MODELS:
        r = q_rows[q_rows[COL_MODEL] == m]
        if len(r):
            FAULTY_QUERIES[m] = r[COL_SQL].iloc[0]



REFERENCE_QUERY = """
SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents FROM orders o JOIN product p ON p.product_id = o.product_id JOIN customers c ON c.customer_id = o.customer_id WHERE (CAST(substr('2026-12-01', 1, 4) AS INTEGER) - CAST(substr(c.birth_date, 1, 4) AS INTEGER) - (substr('2026-12-01', 6, 5) < substr(c.birth_date, 6, 5))) = 25;
"""

for m, sql in FAULTY_QUERIES.items():
    print(f"===== {m} =====\n{sql}\n")

===== cohere/north-mini-code:free =====
SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN product p ON o.product_id = p.product_id
WHERE
  -- Compute age on 2026-12-01
  (
    -- Age = year difference
    CAST(SUBSTR('2026-12-01', 1, 4) AS INTEGER) -
    CAST(SUBSTR(c.birth_date, 1, 4) AS INTEGER)
  ) =
  CASE
    -- If birthday hasn't occurred yet this year, subtract 1
    WHEN SUBSTR(c.birth_date, 6, 2) > SUBSTR('2026-12-01', 6, 2) THEN 1
    WHEN SUBSTR(c.birth_date, 6, 2) = SUBSTR('2026-12-01', 6, 2) AND SUBSTR(c.birth_date, 9, 2) > SUBSTR('2026-12-01', 9, 2) THEN 1
    ELSE 0
  END
  AND
  -- Ensure the birthday has occurred on or before 2026-12-01
  (
    (SUBSTR(c.birth_date, 6, 2) < SUBSTR('2026-12-01', 6, 2)) OR
    (SUBSTR(c.birth_date, 6, 2) = SUBSTR('2026-12-01', 6, 2) AND SUBSTR(c.birth_date, 9, 2) <= SUBSTR('2026-12-01', 9, 2))
  );

===== dots-studio/dots-3-note-preview:free =====
S

In [104]:
con = sqlite3.connect(DB_PATH)


def run(sql):
    return pd.read_sql_query(sql, con)


for m, sql in FAULTY_QUERIES.items():
    out = run(sql)
    print(f"{m}: {out.iloc[0, 0]!r}  (expected wrong value: {WRONG_VALUE})")

if "..." not in REFERENCE_QUERY:
    ref_out = run(REFERENCE_QUERY).iloc[0, 0]
    print(f"Reference query: {ref_out!r}  (reference-td1.csv: {REFERENCE})")
    assert normalize(ref_out) == REFERENCE, "Your reference query does not give the CSV value!"

cohere/north-mini-code:free: np.int64(0)  (expected wrong value: 0)
dots-studio/dots-3-note-preview:free: np.int64(0)  (expected wrong value: 0)
Reference query: np.int64(446690)  (reference-td1.csv: 446690)


In [105]:
# Useful for the diagnosis: schema and format of the columns involved
for t in ("customers", "orders", "product"):
    print(t, run(f"PRAGMA table_info({t})")[["name", "type"]].values.tolist())
display(run("SELECT MIN(birth_date) AS min_bd, MAX(birth_date) AS max_bd, COUNT(*) AS n FROM customers"))
# Customers born around the window [2000-12-02, 2001-12-01] (= exactly 25 years old on 2026-12-01)
display(run("""SELECT customer_id, birth_date FROM customers
               WHERE birth_date BETWEEN '2000-11-01' AND '2001-12-31' ORDER BY birth_date"""))
# What each faulty WHERE clause actually selects (customers, before any join)
print("Customers born in 2026 (only ones cohere's clause can keep):",
      run("SELECT COUNT(*) FROM customers WHERE birth_date LIKE '2026%'").iloc[0, 0])
print("Customers born exactly 2001-12-01 (only ones dots' clause can keep):",
      run("SELECT COUNT(*) FROM customers WHERE birth_date = '2001-12-01'").iloc[0, 0])
print("SQLite check of the '25 years' modifier:",
      run("SELECT strftime('%Y-%m-%d', '2001-12-01', '25 years') AS x").iloc[0, 0])

customers [['customer_id', 'TEXT'], ['insert_date', 'TEXT'], ['birth_date', 'TEXT']]
orders [['date', 'TEXT'], ['order_id', 'TEXT'], ['customer_id', 'TEXT'], ['product_id', 'TEXT'], ['quantity_product', 'INTEGER']]
product [['product_id', 'TEXT'], ['price', 'INTEGER']]


,min_bd,max_bd,n
0,1958-06-08,2006-08-15,59


,customer_id,birth_date
0,C007,2000-11-30
1,C009,2001-03-10
2,C008,2001-05-15
3,C002,2001-09-24
4,C001,2001-09-25
5,C003,2001-09-26
6,C006,2001-11-30
7,C014,2001-12-08


Customers born in 2026 (only ones cohere's clause can keep): 0
Customers born exactly 2001-12-01 (only ones dots' clause can keep): 0
SQLite check of the '25 years' modifier: 2026-12-01


## 5. Diagnosis of the faulty clause

Question C11: revenue excluding VAT generated by customers **exactly 25 years old on 2026-12-01**,
i.e. born between **2000-12-02 and 2001-12-01 inclusive**. Reference: 6 customers
(C001, C002, C003, C006, C008, C009) → **446 690 cents**. C007 (born 2000-11-30) is the boundary trap:
already 26 on 2026-12-01, so correctly excluded.

**cohere/north-mini-code:free**
- Clause at fault: `(year(2026-12-01) - year(birth_date)) = CASE WHEN <birthday not reached yet> THEN 1 ELSE 0 END AND <birthday already reached>`.
  The model computed the "birthday not yet passed" correction (0 or 1) correctly, but instead of *subtracting* it from the
  year difference and comparing the result to **25**, it compared the year difference **to the correction itself**.
  The number 25 never appears in the query.
- Rows actually selected: year difference = 0 *and* birthday reached ⇒ customers born in **2026, on or before 1 December**
  → 0 customers in the table (birth dates range 1958-06-08 … 2006-08-15).
- Rows that should be selected: born 2000-12-02 … 2001-12-01 (6 customers).
- Why 0: empty set after WHERE ⇒ `SUM` is NULL ⇒ `COALESCE(..., 0)` returns 0. The query is valid SQL and executes cleanly,
  so nothing signals the error.

**dots-studio/dots-3-note-preview:free**
- Clause at fault: `strftime('%Y-%m-%d', c.birth_date, '25 years') = '2026-12-01'`.
  It tests "the 25th birthday falls **exactly on** 2026-12-01" (equality) instead of "the 25th birthday is **on or before**
  2026-12-01 and the 26th is after" (an interval). It confuses *being 25 on a date* with *turning 25 on that date*.
- Rows actually selected: only customers born on **2001-12-01** → 0 customers in the table.
- Rows that should be selected: the full one-year window 2000-12-02 … 2001-12-01 (6 customers).
- Why 0: same mechanism, empty set + `COALESCE` → 0.

**Same reason or two different reasons?** Two **different** SQL errors (cohere: wrong comparison operand, the age is
compared to the correction term instead of to 25; dots: equality on a single birthday instead of a one-year interval).
They agree on 0 only because both collapse the selection to an **empty set** and both use `COALESCE(SUM(...), 0)`:
0 is the default value of any query that selects nothing. Agreement on 0 is therefore weak evidence — two independent
bugs land on the same "attractor" value. Note also that Dots is not stable: its other run (traces-E-Mathias.csv) returned
the correct 446 690.

## 6. Delta query

List the `customer_id`s (and their revenue excluding VAT) counted by the reference but missed
(or wrongly added) by the faulty query. The sum must equal `GAP_CENTS` **to the cent**.

Typical pattern: *reference condition* `AND NOT (` *faulty condition* `)`, grouped by customer.

In [106]:
# Reference condition (exactly 25 on 2026-12-01) AND NOT (faulty condition), grouped by customer.
REF_COND = """(CAST(substr('2026-12-01', 1, 4) AS INTEGER) - CAST(substr(c.birth_date, 1, 4) AS INTEGER)
     - (substr('2026-12-01', 6, 5) < substr(c.birth_date, 6, 5))) = 25"""

FAULTY_COND = {
    "cohere/north-mini-code:free": """(
        (CAST(SUBSTR('2026-12-01', 1, 4) AS INTEGER) - CAST(SUBSTR(c.birth_date, 1, 4) AS INTEGER)) =
        CASE WHEN SUBSTR(c.birth_date, 6, 2) > SUBSTR('2026-12-01', 6, 2) THEN 1
             WHEN SUBSTR(c.birth_date, 6, 2) = SUBSTR('2026-12-01', 6, 2)
                  AND SUBSTR(c.birth_date, 9, 2) > SUBSTR('2026-12-01', 9, 2) THEN 1
             ELSE 0 END
        AND ((SUBSTR(c.birth_date, 6, 2) < SUBSTR('2026-12-01', 6, 2)) OR
             (SUBSTR(c.birth_date, 6, 2) = SUBSTR('2026-12-01', 6, 2)
              AND SUBSTR(c.birth_date, 9, 2) <= SUBSTR('2026-12-01', 9, 2))))""",
    "dots-studio/dots-3-note-preview:free": """(c.birth_date IS NOT NULL
        AND strftime('%Y-%m-%d', c.birth_date, '25 years') = '2026-12-01')""",
}


def delta_query(faulty_cond):
    return f"""
SELECT c.customer_id, c.birth_date,
       SUM(o.quantity_product * p.price) AS revenue_cents
FROM orders o
JOIN product p   ON p.product_id = o.product_id
JOIN customers c ON c.customer_id = o.customer_id
WHERE {REF_COND}
  AND NOT {faulty_cond}
GROUP BY c.customer_id, c.birth_date
ORDER BY c.customer_id;
"""


DELTA_QUERY = delta_query(FAULTY_COND["cohere/north-mini-code:free"])

for m, cond in FAULTY_COND.items():
    # Opposite direction too: rows counted by the faulty query but not by the reference (should be none)
    extra = run(f"""SELECT COUNT(*) FROM orders o JOIN customers c ON c.customer_id = o.customer_id
                    WHERE {cond} AND NOT {REF_COND}""").iloc[0, 0]
    delta = run(delta_query(cond))
    print(f"===== {m} =====")
    display(delta)
    delta_sum = int(delta.iloc[:, -1].sum())   # last column = revenue
    print(f"Missed revenue = {delta_sum} cents | wrongly added order lines = {extra} | expected gap = {GAP_CENTS} cents")
    print("Gap fully explained" if delta_sum == GAP_CENTS and extra == 0
          else f"Unexplained difference: {GAP_CENTS - delta_sum} cents")
    print()

===== cohere/north-mini-code:free =====


,customer_id,birth_date,revenue_cents
0,C001,2001-09-25,135000
1,C002,2001-09-24,25920
2,C003,2001-09-26,47900
3,C006,2001-11-30,77450
4,C008,2001-05-15,99090
5,C009,2001-03-10,61330


Missed revenue = 446690 cents | wrongly added order lines = 0 | expected gap = 446690 cents
Gap fully explained

===== dots-studio/dots-3-note-preview:free =====


,customer_id,birth_date,revenue_cents
0,C001,2001-09-25,135000
1,C002,2001-09-24,25920
2,C003,2001-09-26,47900
3,C006,2001-11-30,77450
4,C008,2001-05-15,99090
5,C009,2001-03-10,61330


Missed revenue = 446690 cents | wrongly added order lines = 0 | expected gap = 446690 cents
Gap fully explained



## 7. Independent check of the reference

Method: recompute in pandas, **without reusing the SQL WHERE clause** nor string arithmetic on dates.
Birth dates are parsed as real dates and the completed age is computed with the textbook rule
`age = year(D) - year(birth) - 1 if (month, day) of D < (month, day) of birth else 0`, then cross-checked with
the equivalent closed window `2000-12-01 < birth_date <= 2001-12-01`. Orders are joined to products in pandas and
`quantity × price` is summed. Both routes must give the CSV reference to the cent.

In [107]:
customers = run("SELECT * FROM customers")
orders = run("SELECT * FROM orders")
product = run("SELECT * FROM product")

AT = pd.Timestamp("2026-12-01")
bd = pd.to_datetime(customers["birth_date"], format="%Y-%m-%d")

# Route 1: completed age computed date by date
customers["age_at"] = [AT.year - b.year - ((AT.month, AT.day) < (b.month, b.day)) for b in bd]
target_age = customers[customers["age_at"] == 25]

# Route 2: closed date window (exactly 25 on 2026-12-01 <=> born in ]2000-12-01, 2001-12-01])
target_win = customers[(bd > "2000-12-01") & (bd <= "2001-12-01")]
assert set(target_age["customer_id"]) == set(target_win["customer_id"]), "The two routes disagree!"

m = orders.merge(target_age, on="customer_id").merge(product, on="product_id")
m["revenue_cents"] = m["quantity_product"] * m["price"]
display(m.groupby(["customer_id", "birth_date"])["revenue_cents"].sum().reset_index())

# Boundary check: neighbours just outside the window must be excluded
display(customers[(bd >= "2000-11-01") & (bd <= "2000-12-31")][["customer_id", "birth_date", "age_at"]])

INDEPENDENT = int(m["revenue_cents"].sum())
if INDEPENDENT is not None:
    print(f"Independent recomputation = {INDEPENDENT} | reference = {REFERENCE}")
    print("Match" if str(INDEPENDENT) == REFERENCE else "Mismatch: check the reference!")

,customer_id,birth_date,revenue_cents
0,C001,2001-09-25,135000
1,C002,2001-09-24,25920
2,C003,2001-09-26,47900
3,C006,2001-11-30,77450
4,C008,2001-05-15,99090
5,C009,2001-03-10,61330


,customer_id,birth_date,age_at
6,C007,2000-11-30,26


Independent recomputation = 446690 | reference = 446690
Match


## 8. DEMO-VOTE (toy example, required)

One-row table: sale on 2026-01-01, 100 cents. Period [2026-01-01, 2026-01-02).

In [108]:
toy = sqlite3.connect(":memory:")
toy.execute("CREATE TABLE sales(sale_date TEXT, amount_cents INTEGER)")
toy.execute("INSERT INTO sales VALUES ('2026-01-01', 100)")

q_strict = """SELECT COALESCE(SUM(amount_cents), 0) FROM sales
              WHERE sale_date >  '2026-01-01' AND sale_date < '2026-01-02'"""
q_incl = """SELECT COALESCE(SUM(amount_cents), 0) FROM sales
            WHERE sale_date >= '2026-01-01' AND sale_date < '2026-01-02'"""

v_strict = toy.execute(q_strict).fetchone()[0]
v_incl = toy.execute(q_incl).fetchone()[0]
print(f"'>'  variant: {v_strict}   (excludes the 1 January sale -> lower boundary wrongly excluded)")
print(f"'>=' variant: {v_incl}   (includes it, matches the reference 100)")

proposals = [v_strict, v_strict, v_incl]
c = Counter(proposals)
winner = next((v for v, n in c.items() if n >= 2), None)
print(f"Proposals {proposals} -> 2-of-3 quorum returns {winner}; reference = 100")
print("=> majority agreement on a wrong value: the quorum validates the error.")

'>'  variant: 0   (excludes the 1 January sale -> lower boundary wrongly excluded)
'>=' variant: 100   (includes it, matches the reference 100)
Proposals [0, 0, 100] -> 2-of-3 quorum returns 0; reference = 100
=> majority agreement on a wrong value: the quorum validates the error.


## 10. Summary to copy into `reports/td2.md`

In [109]:
print(f"""
## Case {QUESTION}
- Source / nature: real batch — own TD 1 traces (traces-td1.csv, nature = real; cohere from traces-C.csv,
  Dots from traces-E.csv, its last attempt on C11)
- Question: revenue excluding VAT of customers exactly 25 years old on 2026-12-01
- Agreeing models: {", ".join(AGREEING_MODELS)}
- Shared wrong value: {WRONG_VALUE}
- Independent reference (reference-td1.csv): {REFERENCE}
- Gap: {GAP_CENTS} cents
- Fixed panel {{ {", ".join(PANEL_MODELS)} }}, k = 2 of n = 3: {VOTE_2_3[1]} -> {VOTE_2_3[2]}
  (cohere votes 0 but gpt-oss-120b and qwen outvote it)
- Quorum {{ {", ".join(ALL_MODELS)} }}, k = 3 of n = 5: {VOTE_3_5[1]} -> {VOTE_3_5[2]}
  (0 gets 2 votes out of 5, below k = 3)
- SQL root cause: two DIFFERENT bugs that both empty the selection, then COALESCE(SUM, 0) returns 0.
  * cohere: compares the year difference to the 'birthday not yet reached' correction (0/1) instead of to 25
    -> keeps only people born in 2026 -> 0 rows.
  * dots: strftime(birth_date, '25 years') = '2026-12-01' -> keeps only people born exactly 2001-12-01
    (25th birthday ON that day instead of 'aged 25 on that day') -> 0 rows.
- Delta query: reference condition AND NOT (faulty condition), grouped by customer; for both models it returns
  C001 135000, C002 25920, C003 47900, C006 77450, C008 99090, C009 61330 = {GAP_CENTS} cents, 0 rows wrongly added
  -> gap explained to the cent.
- Independent checking method: pandas recomputation with parsed dates (completed age rule, cross-checked with the
  window ]2000-12-01, 2001-12-01]) = {INDEPENDENT} cents = reference. In production the cheap detector is a
  sanity rule: a 0 coming from COALESCE on an empty selection must trigger a 'how many customers matched?' count
  query (here 0 vs 6) before the value is accepted. Neither quorum was fooled here, but this is luck of the
  panel composition, not a guarantee: a 2-of-3 panel {{cohere, dots, X}} would have returned 0.
""")


## Case C11
- Source / nature: real batch — own TD 1 traces (traces-td1.csv, nature = real; cohere from traces-C.csv,
  Dots from traces-E.csv, its last attempt on C11)
- Question: revenue excluding VAT of customers exactly 25 years old on 2026-12-01
- Agreeing models: cohere/north-mini-code:free, dots-studio/dots-3-note-preview:free
- Shared wrong value: 0
- Independent reference (reference-td1.csv): 446690
- Gap: 446690 cents
- Fixed panel { openai/gpt-oss-120b, cohere/north-mini-code:free, qwen/qwen3.8-27b }, k = 2 of n = 3: 446690 -> the REFERENCE wins (wrong value outvoted)
  (cohere votes 0 but gpt-oss-120b and qwen outvote it)
- Quorum { cohere/north-mini-code:free, dots-studio/dots-3-note-preview:free, openai/gpt-oss-120b, openai/gpt-oss-20b, qwen/qwen3.8-27b }, k = 3 of n = 5: 446690 -> the REFERENCE wins (wrong value outvoted)
  (0 gets 2 votes out of 5, below k = 3)
- SQL root cause: two DIFFERENT bugs that both empty the selection, then COALESCE(SUM, 0) returns 0.
  * cohe